# Build FAISS Wikipedia index (Colab GPU)

One-shot job. Pick **Runtime → Change runtime type → T4 GPU**, run all cells, then download `wiki_subset.faiss` and `wiki_subset.jsonl` from `/content/data/index/` and place them under `data/index/` in your local repo.

Wall time: ~15–30 min for 200k passages with `bge-small-en-v1.5` on a T4.

In [1]:
!pip -q install "datasets<4.0.0" sentence-transformers faiss-cpu tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 77.7 MB/s eta 0:00:00


In [2]:
import datasets
# make sure it's 3.x.x
print(datasets.__version__)

3.6.0


In [3]:
# Pull the build_index module straight from your repo, or copy-paste the function.
# Easiest: clone the repo here on Colab.
# !git clone https://github.com/<your-fork>/adaptive-rag.git
# %cd adaptive-rag
# Otherwise, paste src/adaptive_rag/corpus/build_index.py contents into the next cell.

In [4]:
"""Build a FAISS index over a Wikipedia passage subset.

Outputs two files in `out_dir`:
  - wiki_subset.faiss   (FAISS index, IndexFlatIP if normalized embeddings)
  - wiki_subset.jsonl   (one passage per line: {doc_id, title, text, url, source})

Designed to run on Colab GPU (one-shot) or locally on CPU. The same script
covers both — set `device='cuda'` on Colab and re-run.

Default corpus: `wiki_dpr` (DPR's Wikipedia passage dump, 100-word chunks).
Truncate to `num_passages` for a manageable subset.
"""

from __future__ import annotations

import json
import os
from pathlib import Path

import numpy as np
from tqdm import tqdm


def build(
    out_dir: str,
    num_passages: int = 200_000,
    embedder_model: str = "BAAI/bge-small-en-v1.5",
    batch_size: int = 64,
    device: str = "cpu",
    hf_dataset: str = "wiki_dpr",
    hf_config: str = "psgs_w100.nq.no_index.no_embeddings",
    seed: int = 42,
) -> dict:
    """Build the index. Returns a dict of paths and stats."""
    import faiss
    from datasets import load_dataset
    from sentence_transformers import SentenceTransformer

    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    passages_path = out / "wiki_subset.jsonl"
    index_path = out / "wiki_subset.faiss"

    print(f"[corpus] Loading {hf_dataset}/{hf_config}...")
    ds = load_dataset(hf_dataset, hf_config, split="train", trust_remote_code=True)

    if num_passages < len(ds):
        # Deterministic subset.
        rng = np.random.default_rng(seed)
        idxs = rng.choice(len(ds), size=num_passages, replace=False)
        idxs.sort()
        ds = ds.select(idxs.tolist())

    print(f"[corpus] Materializing {len(ds)} passages to {passages_path}")
    with passages_path.open("w", encoding="utf-8") as fh:
        for i, row in enumerate(tqdm(ds, desc="passages")):
            payload = {
                "doc_id": str(row.get("id", i)),
                "title": str(row.get("title", "")),
                "text": str(row.get("text", "")),
                "source": "wiki",
            }
            fh.write(json.dumps(payload) + "\n")

    print(f"[corpus] Loading embedder {embedder_model} on {device}")
    model = SentenceTransformer(embedder_model, device=device)
    dim = model.get_sentence_embedding_dimension()
    index = faiss.IndexFlatIP(dim)  # cosine sim with normalized embeddings

    texts = [str(row.get("text", "")) for row in ds]
    n = len(texts)
    print(f"[corpus] Embedding {n} passages (batch_size={batch_size})")
    for start in tqdm(range(0, n, batch_size), desc="embed"):
        batch = texts[start : start + batch_size]
        emb = model.encode(
            batch,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False,
            batch_size=batch_size,
        ).astype(np.float32)
        index.add(emb)

    faiss.write_index(index, str(index_path))
    print(f"[corpus] Wrote index to {index_path} (n={index.ntotal}, dim={dim})")

    return {
        "index_path": str(index_path),
        "passages_path": str(passages_path),
        "num_passages": n,
        "embedder": embedder_model,
        "dim": dim,
    }

In [5]:
import sys
sys.path.insert(0, 'src')

info = build(
    out_dir='/content/data/index',
    num_passages=200_000,
    embedder_model='BAAI/bge-small-en-v1.5',
    batch_size=128,
    device='cuda',
)
info

[corpus] Loading wiki_dpr/psgs_w100.nq.no_index.no_embeddings...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

wiki_dpr.py: 0.00B [00:00, ?B/s]

data/psgs_w100/no_embeddings/train-00000(…):   0%|          | 0.00/282M [00:00<?, ?B/s]

...

/tmp/ipykernel_7767/501891537.py:67: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dim = model.get_sentence_embedding_dimension()


[corpus] Embedding 200000 passages (batch_size=128)


embed: 100%|██████████| 1563/1563 [14:02<00:00,  1.85it/s]


[corpus] Wrote index to /content/data/index/wiki_subset.faiss (n=200000, dim=384)


{'index_path': '/content/data/index/wiki_subset.faiss',
 'passages_path': '/content/data/index/wiki_subset.jsonl',
 'num_passages': 200000,
 'embedder': 'BAAI/bge-small-en-v1.5',
 'dim': 384}

In [6]:
# Download the artifacts.
from google.colab import files
files.download('/content/data/index/wiki_subset.faiss')
files.download('/content/data/index/wiki_subset.jsonl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>